# 04 — QLoRA ile Instruction Fine-Tuning

**Kapsam:** Transformer tabanlı büyük dil modelleri üzerinde eğitim; LoRA, QLoRA, PEFT
ve instruction tuning teknikleri.

## Neden QLoRA?
- **Full fine-tuning**: Tüm parametreleri günceller. Milyarlarca parametre için Colab'ın
  ücretsiz T4 GPU'suna (~15GB VRAM) sığmaz.
- **LoRA**: Temel modeli dondurur, her katmana küçük "adaptör" matrisleri ekler; sadece
  bunları eğitir (parametrelerin <%1'i).
- **QLoRA**: LoRA'nın üstüne, temel modeli 4-bit'e (NF4) sıkıştırarak bellek ihtiyacını
  daha da azaltır — böylece 3B-7B modeller tek bir T4'e sığar.

Bu notebook, 01. notebook'ta ürettiğiniz `sft_dataset.jsonl` (taslak notlar → düzgün
doküman çiftleri) ile devam eder.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys
os.environ.setdefault("USE_TF", "0")  # transformers TensorFlow'u hic denemesin (Colab'da protobuf catismasi yasatiyor)

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data/raw", "data/processed", "models", "mlruns"]  # chroma_db BILEREK haric (asagida)

try:
    from google.colab import drive
    # force_remount=True KULLANMIYORUZ: bu, zaten mount edilmişken bile her seferinde
    # yeniden yetkilendirme (izin penceresi) ister, gereksiz bekleme/kesinti yaratır.
    # drive.mount() zaten mount edilmişse kendi içinde anında geri döner; mount
    # edilmemişse (bu runtime'da ilk çalıştırma) normal şekilde izin ister — bu
    # durumda çıkan izin penceresini/bağlantısını tamamlamanız gerekir, hücreyi
    # durdurmayın.
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)
        os.makedirs(os.path.dirname(_local_path), exist_ok=True)  # orn. data/ klasorunu gercek dizin olarak olustur

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)
    print("Not: chroma_db (vektor veritabani) Drive'a BAGLANMADI -- SQLite, Drive'in")
    print("     FUSE dosya sisteminde yazma kilidini desteklemiyor ('OperationalError:")
    print("     attempt to write a readonly database'). Her yeni runtime'da RAG")
    print("     notebook'undaki (03) indeksleme hucresini tekrar calistirin -- chunks.jsonl")
    print("     zaten Drive'da oldugu icin bu hizli ve ucretsiz bir islemdir.")


## Colab ortam düzeltmeleri

Kurulum hücresinden hemen sonra çalıştırın. Paket sürümlerini sabitler.

In [ ]:
import os, subprocess, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
bootstrap = os.path.join(PROJECT_DIR, "scripts", "colab_bootstrap.py")

if not os.path.exists(bootstrap):
    raise FileNotFoundError(
        "scripts/colab_bootstrap.py bulunamadi. "
        "Guncel projeyi zip'leyip Drive'a yukleyin."
    )

subprocess.run([sys.executable, bootstrap], check=True, cwd=PROJECT_DIR)

### ⚠️ Kurulum sonrası zorunlu: Runtime yeniden başlatma

`colab_bootstrap.py` numpy/scipy dahil birçok paketi force-reinstall ediyor. Colab'ın
kernel'i numpy'yi bu hücre çalışmadan ÖNCE zaten bellekte tutuyor olabileceği için,
yeniden başlatmadan devam ederseniz sonraki hücrelerde
`ImportError: cannot import name '_center' from numpy._core.umath` hatası alabilirsiniz.

1. **Runtime → Oturumu yeniden başlat** (yukarıdaki kurulum hücresini TEKRAR ÇALIŞTIRMAYIN)
2. Bu notebook'un en üstteki (Drive/symlink) hücresini tekrar çalıştırın
3. Aşağıdaki doğrulama hücresini çalıştırıp devam edin

In [ ]:
import numpy as np
from numpy._core import strings  # eskiden "_center" ImportError'i buradan geliyordu
from transformers.models.qwen2 import modeling_qwen2  # bu projenin gercek modeli -- scipy/numpy ABI sorunlari (orn. "_blas_supports_fpe") transformers'in object-detection loss modulu uzerinden burada ortaya cikardi
print("numpy:", np.__version__, "OK -- devam edebilirsiniz.")

In [ ]:
import torch
print("CUDA kullanılabilir mi:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU bulunamadı. Runtime > Değiştir çalışma zamanı türü > T4 GPU seçin.")


## Hiperparametreler

`src/config.py` içindeki `QLoraConfig`'i inceleyin: `lora_r`, `lora_alpha`, `target_modules` gibi değerler burada tanımlı.

In [ ]:
from src.config import QLORA_CONFIG
print(QLORA_CONFIG)


## Eğitim

Bu hücre GPU'da birkaç dakika ile birkaç saat arasında sürebilir (veri seti boyutuna bağlı).

In [ ]:
from src.finetune.qlora_train import train

adapter_path = train()
print("LoRA adaptörü kaydedildi ->", adapter_path)


## Hızlı doğrulama

Eğitilen adaptörle RAG pipeline'ını tekrar çalıştırıp, temel modelle karşılaştırın (bkz. 07. notebook A/B test).

In [ ]:
import json
from src.rag.vector_store import get_collection, index_chunks

# chroma_db Drive'a baglanmiyor (bkz. en ustteki hucrenin uyarisi), yani bu runtime'da
# 03. notebook'un indeksleme adimi hic calismadiysa koleksiyon bos olur ve asagidaki
# answer() cagrisi "Bu bilgi elimdeki dokumanlarda yok." fallback'ini doner. Bos ise
# burada kendi kendine tekrar indeksliyoruz.
if get_collection().count() == 0:
    print("Chroma indeksi bu runtime'da bos -- yeniden indeksleniyor...")
    with open("data/processed/chunks.jsonl", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f]
    index_chunks(chunks)
    print(f"{len(chunks)} chunk indekslendi.")

In [ ]:
from src.rag.rag_pipeline import answer

result = answer("Bir sorun giderme rehberi nasıl yapılandırılmalı?", model_path=adapter_path)
print(result["answer"])
